# 32 One-vs-Rest Coherent LR Projection

Project raw one-vs-rest outputs to Bayes-coherent multiclass one-vs-rest LRs.
Raw outputs remain unchanged; coherent outputs are written to a separate root.


In [ ]:
from pathlib import Path
import subprocess
import sys

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

REPO_ROOT = _find_repo_root()
REPO_ROOT


In [ ]:
# Runtime controls
MODEL_ID = "gpt-5.3-chat-latest"
SCENARIO_FILTER = None  # e.g., ["chest_pain_carter_8"]
MAX_SCHEMAS = None
MAX_FINDINGS = None
REG = 1e-6
FAIL_ON_MISSING_PRIORS = False
OVERWRITE = True
WRITE_POSTERIOR_DEBUG = False

# Backward-compatibility helper for historical runs:
# when schema_priors.csv is missing, derive it from existing inputs/source sheets
# without rerunning one-vs-rest LLM estimation.
DERIVE_PRIORS_IF_MISSING = True


In [ ]:
cmd = [
    "uv", "run", "--group", "notebooks", "python",
    "scripts/project_one_vs_rest_coherent_lrs.py",
    "--model-id", MODEL_ID,
    "--reg", str(REG),
]

if SCENARIO_FILTER:
    for sid in SCENARIO_FILTER:
        cmd.extend(["--scenario-filter", sid])
if MAX_SCHEMAS is not None:
    cmd.extend(["--max-schemas", str(MAX_SCHEMAS)])
if MAX_FINDINGS is not None:
    cmd.extend(["--max-findings", str(MAX_FINDINGS)])
if FAIL_ON_MISSING_PRIORS:
    cmd.append("--fail-on-missing-priors")
if OVERWRITE:
    cmd.append("--overwrite")
if WRITE_POSTERIOR_DEBUG:
    cmd.append("--write-posterior-debug")
if DERIVE_PRIORS_IF_MISSING:
    cmd.append("--derive-priors-if-missing")

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Coherence projection failed with exit code {result.returncode}.")


In [ ]:
import pandas as pd

manifests = REPO_ROOT / "data/processed/lr_one_vs_rest/manifests"
summary_path = manifests / f"coherence_projection_summary_{MODEL_ID}.csv"
top_rows_path = manifests / f"coherence_projection_top_rows_{MODEL_ID}.csv"
failures_path = manifests / f"coherence_projection_failures_{MODEL_ID}.csv"

print({
    "summary": str(summary_path),
    "top_rows": str(top_rows_path),
    "failures": str(failures_path),
})

if summary_path.exists():
    display(pd.read_csv(summary_path))
if failures_path.exists():
    failures_df = pd.read_csv(failures_path)
    print(f"failures={len(failures_df)}")
    if len(failures_df):
        display(failures_df.head(20))
